**Author:** Amanda Baright

**Date:** 04.14.2026

**Purpose:** ST 554 Homework 9

# Model Fitting with `pyspark`

## Data Introduction

This homework assignment will focus on using `pyspark` and `Spark MLlib` to fit three different classes of models using a pipline approach. For this assignmnet, I'll be using the *Maternal Health Risk* dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/863/maternal+health+risk).

Maternal mortality is one of the main concerns for the Sustainable Development Goals of the UN, and continues to be a pressing global concern. The data is collected from different hospitals, community clinics and maternal health cares from rural Bangladesh, and features 1013 subjects with 6 covariate measurements (`Age`, Systolic Blood Pressure as `SystolicBP`, Diastolic BP as `DiastolicBP`, Blood Sugar as `BS`, Body Temperature as `BodyTemp`, `HeartRate`) and the response `RiskLevel` (that is categorical and has 3 levels - `low risk`, `mid risk`, and `high risk`). All of these are the responsible and significant risk factors for maternal mortality and the `RiskLevel` variable. For the purposes of this assignment we will group the `mid risk` and `high risk` categories to ensure we have a binary response.

We will first want to start our Spark session.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 18:33:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 18:33:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 18:33:58 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


We now want to read in the data set using `pandas` and convert it to a `spark` SQL style data frame.

In [2]:
health_data = pd.read_csv("Maternal Health Risk Data Set.csv")
health_data.head()

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


Now we convert it to a `spark` SQL style data frame.

In [3]:
health = spark.createDataFrame(health_data)
health.show(5)

+---+----------+-----------+----+--------+---------+---------+
|Age|SystolicBP|DiastolicBP|  BS|BodyTemp|HeartRate|RiskLevel|
+---+----------+-----------+----+--------+---------+---------+
| 25|       130|         80|15.0|    98.0|       86|high risk|
| 35|       140|         90|13.0|    98.0|       70|high risk|
| 29|        90|         70| 8.0|   100.0|       80|high risk|
| 30|       140|         85| 7.0|    98.0|       70|high risk|
| 35|       120|         60| 6.1|    98.0|       76| low risk|
+---+----------+-----------+----+--------+---------+---------+
only showing top 5 rows


We now want to create the binary version of `RiskLevel` where we will combine `mid risk` and `high risk` to be one category. Here `low risk` will be `0` and the others will be `1`. Here we will go ahead and label the binary version as `label` to help set us up for the `spark MLlib` model fitting steps.

In [4]:
from pyspark.sql.functions import when, col
health = health.withColumn("label",
                           when(col("RiskLevel") == "low risk", 0).otherwise(1))
health.show(5)

+---+----------+-----------+----+--------+---------+---------+-----+
|Age|SystolicBP|DiastolicBP|  BS|BodyTemp|HeartRate|RiskLevel|label|
+---+----------+-----------+----+--------+---------+---------+-----+
| 25|       130|         80|15.0|    98.0|       86|high risk|    1|
| 35|       140|         90|13.0|    98.0|       70|high risk|    1|
| 29|        90|         70| 8.0|   100.0|       80|high risk|    1|
| 30|       140|         85| 7.0|    98.0|       70|high risk|    1|
| 35|       120|         60| 6.1|    98.0|       76| low risk|    0|
+---+----------+-----------+----+--------+---------+---------+-----+
only showing top 5 rows


## Splitting the Data

Now we will want to split the data into a training and test set. This can be done using the `.randomSplit()` method on a Spark SQL style data frame. Setting up the data split before any transformations is critical, as we want to ensure that any transformations done on the training set are done on the test set.

In [17]:
train, test = health.randomSplit([0.8,0.2], seed = 1)
print(train.count(), test.count())

807 207


## Metric Choice: Log Loss

To evaluate the performance of our maternal health risk models, I have selected Log Loss as the primary evaluation metric.

While metrics like Accuracy simply count whether a model got a prediction "right" or "wrong", Log Loss measures the uncertainty of the model's predictions. It looks at the probability assigned to the correct class. If the model is very confident in a correct prediction, the Log Loss is low. If the model is confident in a prediction that turns out to be wrong, the Log Loss increases exponentially.

I've choosen Log Loss over the common RMSE (Root Mean Squared Error) metric used in the course, as the RMSE metric treats the distance between 0 and 1 as a simple linear gap. Log Loss, however, uses a logarithmic scale. This means that if the model predicts a near-zero probability of risk for a patient who is actually high-risk, the metric will reflect a massive failure. In a clinical setting, an overconfident "miss" is much more dangerous than a "maybe."

This choice also aligns more with Classification of our binary response variable, as the Log Loss is the natural "likelihood' function used in Logistic Regression. Using it as our evaluation metric ensures that our cross-validation process is optimizing for the true probabilistic nature of the data rather than treating the 0s and 1s as simple numeric distances. When evaluating the metric, our objective is to find the model that achieves the **lowest** possible Log Loss on the unseen test data, indicating that the model is both accurate and well-calibrated.

## Metric Choice: Area Under the Curve

To evaluate the performance of our maternal health risk models, I have selected Area Under the Curve (AUC) as the primary evaluation metric.

AUC measures the model's ability to discriminate between classes. Specifically, it represents the probability that the model will rank a randomly chosen "risk" patient higher than a randomly chosen "low risk" patient.

In a medical context like maternal health, we are highly concerned with the tradeoff between Sensitivity (finding all at-risk patients) and Specificity (not alarming healthy patients). AUC provides a single aggregate measure of performance across all possible classification thresholds. Unlike Accuracy and other metrics, AUC is robust to class imbalance and tells us how well our features actually "seperate" the two risk groups.

When it comes to interpreting AUC, an AUC greater than 1.0 is a perfect model and an AUC around 0.5 is no better than random guessing. The goal is to maximize the AUC, so higher values are better.

## Three Model Choices

### Model 1: Logistic Regression

The first model we will look at is a Logistic Regression model, which is used as a functional tool for binary classification. Logistic Regression is an extension of a Generalized Linear Model (GLM). In standard regression, we often predict a continuous value. However, in Logistic Regression, we apply a "link function" like the logit to a linear combination of the predictors.

This model will then map any predicted value into a probability between 0 and 1. Statistically, it is estimating the log-odds that a patient belongs to the "risk" category. The model assums a linear relationship between the independent variables and the log-odds of the response exsits. The advantage for this model is that it is highly interpretable, just like a T-Test or ANOVA, where we can see exactly which health vitals has the strongest positive or negative correlation with the risk level.

### Model 2: Random Forest Classifier

The next model we might want to look at is a Random Forest with classification trees. To understand a Random Forest, you must first understand a Decision Tree. A Decision Tree is a non-parametric model that partitions the data into smaller and smaller subgroups based on feature thresholds (like Age > 35). However, a single tree is prone to high variance and can be prone to overfitting (i.e. being too attached to the specific noise in the training data).

A Random Forest then is an ensemble of many decision trees. It uses a statistical technique called Bootstrap Aggregating (or Bagging), and creates many different versions of the dataset through resempling with replacement and builds a tree for each version. When it's time to predict a patient's risk, the model averages these results. The Random Forest cancels out the individual errors of the trees, resulting in a model that is more robust and better at generalizing to new patients.

### Model 3: Gradient Boosted Trees (GBT)

Similar to a Random Forest, GBT is an ensemble of decision trees. However, the statistical ideas behind the algorithm are different. While Random Forest builds trees in paralllel to reduce variation, GBT builds them sequentially to reduce bias.

The GBT model uses an iterative process called boosting, where the first tree makes a simple prediction. The second tree is then trained specifically to predict the residuals of the first tree. Each subsequent tree tries to correct the mistakes made by the combination of all previous trees. In statistical terms, it is performing a gradient descent in the functional space to minimize our Log Loss. This often makes GBT the most accurate of the three models, though it requires more careful tuning to ensure it doesn't have bias to any outliers present in the data.

## Model Fitting

Now it's time to do some model fitting! We first want to import any packages we need.

In [32]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (VectorAssembler, StandardScaler, 
                                SQLTransformer, PolynomialExpansion)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

### Transformations

Next we want to start defining our pipeline and develop our transformations. Here we can use the same set of transformations for each model. 

#### Transformation 1

We can use `SQLTransformer()` to create a new variable called `PulsePressure` that subtracts the difference between Systolic and Diastolic BP. We can then use the `.transform()` method to see this new variable.

In [7]:
sql_trans = SQLTransformer(
    statement="SELECT *, (SystolicBP - DiastolicBP) as PulsePressure FROM __THIS__"
)
sql_trans.transform(health).show(5)

+---+----------+-----------+----+--------+---------+---------+-----+-------------+
|Age|SystolicBP|DiastolicBP|  BS|BodyTemp|HeartRate|RiskLevel|label|PulsePressure|
+---+----------+-----------+----+--------+---------+---------+-----+-------------+
| 25|       130|         80|15.0|    98.0|       86|high risk|    1|           50|
| 35|       140|         90|13.0|    98.0|       70|high risk|    1|           50|
| 29|        90|         70| 8.0|   100.0|       80|high risk|    1|           20|
| 30|       140|         85| 7.0|    98.0|       70|high risk|    1|           55|
| 35|       120|         60| 6.1|    98.0|       76| low risk|    0|           60|
+---+----------+-----------+----+--------+---------+---------+-----+-------------+
only showing top 5 rows


#### Transformation 2

Next we want to use `VectorAssmebler` to combine our features into a single feature column called `features`.

In [8]:
assembler = VectorAssembler(
    inputCols=["Age", "SystolicBP", "DiastolicBP", "BS", "BodyTemp", "HeartRate", "PulsePressure"],
    outputCol="features"
)

In [9]:
# Stack the transforms
assembler.transform(
    sql_trans.transform(health)).show(5)

+---+----------+-----------+----+--------+---------+---------+-----+-------------+--------------------+
|Age|SystolicBP|DiastolicBP|  BS|BodyTemp|HeartRate|RiskLevel|label|PulsePressure|            features|
+---+----------+-----------+----+--------+---------+---------+-----+-------------+--------------------+
| 25|       130|         80|15.0|    98.0|       86|high risk|    1|           50|[25.0,130.0,80.0,...|
| 35|       140|         90|13.0|    98.0|       70|high risk|    1|           50|[35.0,140.0,90.0,...|
| 29|        90|         70| 8.0|   100.0|       80|high risk|    1|           20|[29.0,90.0,70.0,8...|
| 30|       140|         85| 7.0|    98.0|       70|high risk|    1|           55|[30.0,140.0,85.0,...|
| 35|       120|         60| 6.1|    98.0|       76| low risk|    0|           60|[35.0,120.0,60.0,...|
+---+----------+-----------+----+--------+---------+---------+-----+-------------+--------------------+
only showing top 5 rows


#### Transform 3

Our next transformation uses `StandardScaler` to Normalize the feature values.

In [10]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")

#### Transform 4

Our final transformation will use `PolynomialExpansion` to add 2 degree interaction terms, which will help capture any non-linear relationship among the health vitals.

In [20]:
poly = PolynomialExpansion(degree=2, inputCol="scaled_features", outputCol="poly_features")

### Model Fitting

Now we can get to actually fitting the models, first starting with our Logistic Regression model. Recall, we will be using Area Under the Curve as the model metric. First we will set up our evaluator using `BinaryClassificationEvaluator`.

In [28]:
evaluator = BinaryClassificationEvaluator(
    labelCol="label", 
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

#### Model 1: Logistic Regression

We will now fit our first model, the Logistic Regression model using the pipeline approach and the four transformations we developed. We then use the `ParamGridBuilder` to help determine if any tuning parameters from regularized or elastic net versions of the logistic regression are needed.

In [29]:
lr = LogisticRegression(featuresCol="poly_features", labelCol="label", probabilityCol="probability")
lr_pipeline = Pipeline(stages=[sql_trans, assembler, scaler, poly, lr])
lr_paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.1, 0.01]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

We then do cross validation using `CrossValidator` for the Logistic Regression Model to determine the best Logistic Regression Model.

In [30]:
cv_lr = CrossValidator(estimator=lr_pipeline, estimatorParamMaps=lr_paramGrid, 
                       evaluator=evaluator, numFolds=5)

We then want to fit this best model on the whole training data.

In [31]:
cv_lr_model = cv_lr.fit(train)

26/04/14 18:52:29 WARN CacheManager: Asked to cache already cached data.
26/04/14 18:52:29 WARN CacheManager: Asked to cache already cached data.


Here we can check which model was the best. We will later use this best model on the test set.

In [34]:
my_list_lr = []
for i in range(len(lr_paramGrid)):
    my_list_lr.append([cv_lr_model.avgMetrics[i], lr_paramGrid[i].values()])
my_list_lr

[[0.8273224396533383, dict_values([0.1, 0.0])],
 [0.8255037843512862, dict_values([0.1, 0.5])],
 [0.8282173775866306, dict_values([0.01, 0.0])],
 [0.8288788070958653, dict_values([0.01, 0.5])]]

### Model 2: Random Forest Classifier

We will now fit the Random Forest Classifier model, again using the pipeline approach and using `ParamGridBuilder` to help select the best tuning parameters.

In [35]:
rf = RandomForestClassifier(featuresCol="poly_features", labelCol="label", probabilityCol = "probability")
rf_pipeline = Pipeline(stages=[sql_trans, assembler, scaler, poly, rf])
rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

We then use `CrossValidator` to perform cross validation.

In [36]:
cv_rf = CrossValidator(estimator=rf_pipeline, estimatorParamMaps=rf_paramGrid, 
                       evaluator=evaluator, numFolds=5)

We then want to fit this best model to the whole training data.

In [37]:
cv_rf_model = cv_rf.fit(train)

We then check to see which model was the best. We will later use this best model on the test set.

In [38]:
my_list_rf = []
for i in range(len(rf_paramGrid)):
    my_list_rf.append([cv_rf_model.avgMetrics[i], rf_paramGrid[i].values()])
my_list_rf

[[0.8840224907465272, dict_values([10, 5])],
 [0.9154825669442914, dict_values([10, 10])],
 [0.8780697113285036, dict_values([20, 5])],
 [0.9233719295776108, dict_values([20, 10])]]

### Model 3: Gradient Boosted Trees

We will now fit our third class of models, a gradient boosted trees model. We again use the pipline approach and use `ParamGridBuilder` to help select the best tuning parameters.

In [39]:
gbt = GBTClassifier(featuresCol="poly_features", labelCol="label")
gbt_pipeline = Pipeline(stages=[sql_trans, assembler, scaler, poly, gbt])
gbt_paramGrid = ParamGridBuilder() \
    .addGrid(gbt.maxIter, [10, 20]) \
    .addGrid(gbt.maxDepth, [3, 5]) \
    .build()

We then use `CrossValidator` to perform cross validation.

In [40]:
cv_gbt = CrossValidator(estimator=gbt_pipeline, estimatorParamMaps=gbt_paramGrid, 
                        evaluator=evaluator, numFolds=5)

We then fit this best model to the whole training data.

In [42]:
cv_gbt_model = cv_gbt.fit(train)

We then check to see which model was the best. We will later use this best model on the test set.

In [43]:
my_list_gbt = []
for i in range(len(gbt_paramGrid)):
    my_list_gbt.append([cv_gbt_model.avgMetrics[i], gbt_paramGrid[i].values()])
my_list_gbt

[[0.883839450369301, dict_values([10, 3])],
 [0.9025400345529834, dict_values([10, 5])],
 [0.8896861457424642, dict_values([20, 3])],
 [0.9090837158631334, dict_values([20, 5])]]

## Model Testing